# Flight Delay Prediction — A Before-Departure Model
### Project introduction & overview

**In one sentence:** given only what is known *before an aircraft departs* — its schedule, carrier, and route — predict how many minutes late it will **arrive**.

This notebook is the front door to the project: it lays out the problem, why it matters, how we formulate it mathematically, the data, the methods, and the plan of attack. It is a **forward-looking overview** — it sets up *what we will do*, not what we found. The work itself lives in companion notebooks — **`Eda.ipynb` → `Feature_Engineering.ipynb` → `Modeling.ipynb`** — and the results are drawn together at the end in **`Conclusion.ipynb`**.

## 1. The problem

A flight's **arrival delay** is the difference, in minutes, between when it actually reaches the gate and when it was scheduled to. We want to **forecast that number ahead of time** — at the moment a passenger might book or check a flight, *before it has pushed back from the gate*.

Formally, we learn a function

$$\hat{y} = f(\mathbf{x}), \qquad \hat{y} \approx \texttt{arr\_delay}\ \text{(minutes)},$$

where the feature vector $\mathbf{x}$ contains **only pre-departure information** (date, scheduled times, carrier, origin, destination, distance, ...). This is a **supervised regression** problem. The insistence on *before departure* is the crux of the project — it is what makes the task both **useful** and **hard** (see §3).

## 2. Why it matters

Delays are a large, chronic cost to the air-travel system:

- **Prevalence.** About **one in five** US domestic flights arrives 15+ minutes late (BTS on-time statistics; ~20.8% on our 2024 data).
- **Cost.** Air-transportation delays cost the US economy on the order of **tens of billions of dollars a year** — a 2010 NEXTOR *Total Delay Impact* study estimated roughly **33 billion dollars** for 2007 alone — spread across airlines (crew, fuel, aircraft utilisation), passengers (lost time, missed connections), and the wider economy.
- **Cascading.** A late aircraft propagates: one delay becomes several downstream, so anticipating delay has outsized value.

**Who benefits from a before-departure forecast?** Passengers plan connections and ground transport; airlines and airports pre-position crews and gates and manage recovery. A forecast is only actionable if it arrives *in advance* — which is precisely why we forbid post-departure information (§3).

**Related work — and the gap we target.** Flight-delay prediction is well studied: Sternberg et al. (2017) survey the field, and models exploiting *network-level* delay structure — clustering by airport, hour, and carrier, and modelling propagation — reach strong accuracy on the same BTS on-time data (e.g. Rebollo & Balakrishnan, 2014). Public notebooks on this very Kaggle dataset likewise report high $R^2$. The catch is that those results lean on **operational / day-of information** — realized departure delays, weather, real-time network congestion — that is *not available before departure*; public notebooks in particular often include `dep_delay`, which **leaks** the outcome. By enforcing a strict pre-departure boundary we deliberately forgo that signal to isolate a narrower, harder, and more honestly-scoped question: **what can the schedule alone predict?**

## 3. The setting: predict *before departure*

This one design choice shapes everything. At prediction time the plane has not moved, so a feature is **legal only if it is knowable at scheduling time**. That rules out a large set of tempting columns the raw data provides but which are recorded *at or after* departure:

- **Departure actuals** — `dep_time`, `dep_delay`, `taxi_out`, `wheels_off`
- **Arrival actuals** — `arr_time`, `taxi_in`, `wheels_on`, `air_time`, `actual_elapsed_time`
- **After-the-fact delay causes** — carrier / weather / NAS / security / late-aircraft minute breakdowns
- **Outcome flags** — `cancelled`, `diverted`, `cancellation_code`

Using any of these would be **data leakage**: the model would "cheat" with information unavailable in production, producing scores that look great in training and collapse on deployment. The whole project is disciplined around this boundary.

**Why not just predict in flight?** Because that is the easy, less-useful version. Once a plane is airborne we already know how late it *left*, and departure delay is by far the strongest driver of arrival delay (the EDA quantifies exactly how strong). A before-departure model must work *without* that dominant signal — which is the real challenge this project takes on.

## 4. The data

- **Source:** Kaggle — [*Flight Data 2024*](https://www.kaggle.com/datasets/hrishitpatil/flight-data-2024) (hrishitpatil), US domestic flights, calendar year **2024**. The records are the standard **US BTS on-time-performance** fields (scheduled / actual times, carrier, route, and the delay cause-code breakdowns).
- **Size:** ~**7.08 million** flights x 35 raw columns.
- **Target:** `arr_delay` — arrival delay in minutes (negative = arrived early).
- After cleaning (removing cancelled/diverted flights, which never produce an arrival to predict), ~**6.97 million** usable flights remain.

The raw file is large (~1.3 GB) and is **git-ignored**; the notebooks read it from `data/raw/` and write cleaned and engineered artifacts to `data/processed/`.

## 5. Formulation & how we measure success

**Target shape.** `arr_delay` is **heavily right-skewed**: most flights land early or on time (median ~ **-6 min**), but a long thin tail of severe delays pulls the mean up (~ **+7 min**), with rare extremes past 60 hours. This shape drives our metric and modelling choices.

**Primary metric — MAE.** For predictions $\hat{y}_i$ and truths $y_i$ over $n$ flights,

$$\mathrm{MAE} = \frac{1}{n}\sum_{i=1}^{n} \lvert y_i - \hat{y}_i \rvert .$$

MAE is **robust to the heavy tail** (a few 60-hour delays don't dominate it) and is directly interpretable ("off by X minutes"). A fact we use repeatedly: the constant that minimises MAE is the **median**, not the mean.

**Secondary — RMSE and $R^2$.**

$$\mathrm{RMSE} = \sqrt{\frac{1}{n}\sum_i (y_i - \hat{y}_i)^2}, \qquad R^2 = 1 - \frac{\sum_i (y_i-\hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}.$$

RMSE penalises large misses quadratically (tail-sensitive); $R^2$ is the fraction of variance explained (1 = perfect, 0 = no better than predicting the mean). Both are reported for context, but since our variance is tail-dominated while we optimise MAE, $R^2$ reads pessimistically low — **MAE stays the headline.**

## 6. Approach — a three-notebook pipeline

| stage | notebook | what happens |
|---|---|---|
| **Explore & clean** | `Eda.ipynb` | quantify missingness (all structural — cancelled/diverted), drop label-less rows, validate values, and profile every candidate feature's relationship to delay |
| **Engineer features** | `Feature_Engineering.ipynb` | derive pre-departure features, **cyclical** time encodings, a **congestion** proxy; a **time-ordered** train/val/test split; **outlier** handling; save one modelling-ready dataset |
| **Model** | `Modeling.ipynb` | a **model ladder** — baselines -> linear (OLS/Ridge) -> **LightGBM** — tuned on validation, tracked with **MLflow** |

Shared, tested helpers (metrics, pipeline builders) live in **`src/`**.

## 7. Methods & key decisions

A few choices are central enough to flag here; each is explained in depth in its own notebook.

- **Leakage-safe, time-aware split.** Because we predict the future, we **sort by time** and split 70 / 10 / 20 into train / validation / test — never a random split, which would leak future flights into training. Everything fit on the target (encoders, the outlier cap) is fit on **train only**, then applied to val/test.
- **Cyclical encoding.** Hour-of-day and day-of-week are *periodic* (23:00 is adjacent to 00:00), so we map each onto a circle via a $\sin/\cos$ pair: $x_{\sin}=\sin(2\pi x/P),\ x_{\cos}=\cos(2\pi x/P)$ for period $P$.
- **High-cardinality categoricals.** Carriers, and especially airports (hundreds of them), are **target-encoded** — each level becomes its mean delay — collapsing many categories to a single informative column without a one-hot explosion.
- **Outliers -> winsorise, not delete.** Extreme delays are *real* (weather meltdowns), and at booking time we cannot know which flights they will be, so we **cap** the training target at the 99th percentile rather than dropping rows (which would bias the model to under-predict).
- **The model ladder.** Trivial baselines set a floor; linear models are interpretable; **gradient-boosted trees** (LightGBM) with an **L1 objective** predict the conditional *median*, matching our MAE goal.

## 8. Reproducing the project

Read/run the notebooks in order — the middle three each write an artifact the next one reads, and `Conclusion.ipynb` draws the results together:

$$\texttt{Eda} \;\longrightarrow\; \texttt{Feature\_Engineering} \;\longrightarrow\; \texttt{Modeling} \;\longrightarrow\; \texttt{Conclusion}$$

**Repository layout**

```
Flight-Delay/
|-- notebooks/
|   |-- Introduction.ipynb         # you are here — overview & plan
|   |-- Eda.ipynb                  # clean + explore  -> data/processed/flights_clean.parquet
|   |-- Feature_Engineering.ipynb  # features + split -> data/processed/flights_features.parquet
|   |-- Modeling.ipynb             # baselines -> linear -> LightGBM
|   +-- Conclusion.ipynb           # synthesis of results & findings
|-- src/
|   |-- metrics.py                 # MAE / RMSE / R^2 helpers
|   +-- models.py                  # preprocessing, pipeline + tuning-objective builders
|-- data/                          # git-ignored (raw + processed)
+-- README.md
```

**Dependencies:** Python 3, `pandas`, `numpy`, `scikit-learn` (>= 1.4), `feature-engine`, `lightgbm`, `mlflow`, `matplotlib`, `pyarrow`.

## 9. References

**Data**
- Patil, H. (2024). *Flight Data 2024* [dataset]. Kaggle. <https://www.kaggle.com/datasets/hrishitpatil/flight-data-2024> — 2024 US domestic on-time records (US BTS-style fields; see the [BTS On-Time Performance documentation](https://www.transtats.bts.gov/) for field definitions).

**Problem & prior work**
- Sternberg, A., Soares, J., Carvalho, D., & Ogasawara, E. (2017). *A Review on Flight Delay Prediction.* arXiv:1703.06118. <https://arxiv.org/abs/1703.06118>
- Rebollo, J. J., & Balakrishnan, H. (2014). *Characterization and prediction of air traffic delays.* Transportation Research Part C, 44, 231–241. <https://doi.org/10.1016/j.trc.2014.04.007>
- Ball, M., Barnhart, C., Dresner, M., et al. (2010). *Total Delay Impact Study.* NEXTOR (the ~\$32.9 B / 2007 estimate). <https://isr.umd.edu/NEXTOR/pubs/TDI_Report_Final_10_18_10_V3.pdf>

**Methods**
- **Gradient boosting** — Friedman, J. H. (2001). *Greedy Function Approximation: A Gradient Boosting Machine.* Annals of Statistics, 29(5), 1189–1232. <https://doi.org/10.1214/aos/1013203451>
- **LightGBM** — Ke, G., Meng, Q., Finley, T., et al. (2017). *LightGBM: A Highly Efficient Gradient Boosting Decision Tree.* NeurIPS. <https://papers.nips.cc/paper/6907-lightgbm-a-highly-efficient-gradient-boosting-decision-tree>
- **Hyperparameter optimization** — Akiba, T., Sano, S., Yanase, T., et al. (2019). *Optuna: A Next-generation Hyperparameter Optimization Framework.* KDD. arXiv:1907.10902. <https://arxiv.org/abs/1907.10902>. TPE sampler: Bergstra, J., Bardenet, R., Bengio, Y., & Kégl, B. (2011). *Algorithms for Hyper-Parameter Optimization.* NeurIPS. <https://dl.acm.org/doi/10.5555/2986459.2986743>
- **Target encoding** — Micci-Barreca, D. (2001). *A Preprocessing Scheme for High-Cardinality Categorical Attributes.* SIGKDD Explorations, 3(1), 27–32. <https://doi.org/10.1145/507533.507538>
- **Cyclical encoding** — feature-engine, `CyclicalFeatures`. <https://feature-engine.trainindata.com/>
- **Equal-variance (reliability) test** — Brown, M. B., & Forsythe, A. B. (1974). *Robust Tests for the Equality of Variances.* JASA, 69(346), 364–367. <https://doi.org/10.1080/01621459.1974.10482955>
- **Outlier rule (modified z-score / MAD)** — Iglewicz, B., & Hoaglin, D. C. (1993). *How to Detect and Handle Outliers.* ASQC Quality Press.
- **Bootstrap confidence intervals** — Efron, B. (1979). *Bootstrap Methods: Another Look at the Jackknife.* Annals of Statistics, 7(1), 1–26. <https://doi.org/10.1214/aos/1176344552>

**Libraries** — [scikit-learn](https://scikit-learn.org/), [LightGBM](https://lightgbm.readthedocs.io/), [feature-engine](https://feature-engine.trainindata.com/), [Optuna](https://optuna.org/), [MLflow](https://mlflow.org/), [SciPy](https://scipy.org/), [pandas](https://pandas.pydata.org/), [NumPy](https://numpy.org/), [Matplotlib](https://matplotlib.org/), [PyArrow](https://arrow.apache.org/docs/python/).